# Carga Relacional en PostgreSQL y Capa Analítica para BI

Este notebook construye la base de datos operacional en PostgreSQL a partir de los archivos depurados en `archivo_final/` y expone tres vistas listas para conectar Power BI o Tableau.

**Arquitectura híbrida:**
* **Tablas físicas (modelo normalizado):** `productos` (31,084 SKUs), `productos_comprados` (5,204 SKUs con movimiento) y `compras` (77,774 renglones efectivos).
* **Vistas analíticas (`vw_bi_*`):** `vw_bi_calendario`, `vw_bi_productos` y `vw_bi_compras`. Resuelven jerarquías de fechas continuas y filtran precios simbólicos sin alterar los importes contables originales.

In [9]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

_cwd = Path.cwd().resolve()
ROOT_DIR = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "archivo_final").is_dir()),
    _cwd,
)

load_dotenv(ROOT_DIR / ".env")

DB_USER = os.getenv("PGUSER", "postgres")
DB_PASS = os.getenv("PGPASSWORD")
DB_HOST = os.getenv("PGHOST", "localhost")
DB_PORT = os.getenv("PGPORT", "5432")
DB_NAME = os.getenv("PGDATABASE", "farmacia_bi")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

df_prod_raw = pd.read_parquet(ROOT_DIR / "archivo_final" / "maestro_productos.parquet")
df_comp_raw = pd.read_parquet(ROOT_DIR / "archivo_final" / "maestro_compras.parquet")

print(f"Maestro productos leído : {len(df_prod_raw):,} filas")
print(f"Maestro compras leído   : {len(df_comp_raw):,} filas")

Maestro productos leído : 31,084 filas
Maestro compras leído   : 77,774 filas


In [10]:
sql_ddl_tablas = """
DROP VIEW IF EXISTS vw_bi_compras, vw_bi_productos, vw_bi_calendario;
DROP TABLE IF EXISTS compras, productos_comprados, productos CASCADE;

CREATE TABLE productos (
    id           SERIAL PRIMARY KEY,
    codigo       BIGINT NOT NULL UNIQUE,
    nombre       VARCHAR(255) NOT NULL,
    droga        VARCHAR(500),
    rubro        VARCHAR(100),
    funcion      VARCHAR(150),
    presentacion VARCHAR(150),
    laboratorio  VARCHAR(150),
    activo       BOOLEAN NOT NULL
);

CREATE TABLE productos_comprados (
    id INTEGER PRIMARY KEY REFERENCES productos(id) ON DELETE CASCADE
);

CREATE TABLE compras (
    id                        SERIAL PRIMARY KEY,
    factura                   VARCHAR(50) NOT NULL,
    producto_id               INTEGER NOT NULL REFERENCES productos(id) ON DELETE RESTRICT,
    codigo                    BIGINT NOT NULL,
    fecha                     TIMESTAMP WITHOUT TIME ZONE NOT NULL,
    cant_ped                  INTEGER,
    cant_fact                 INTEGER,
    cant_entr                 INTEGER,
    precio_unitario           NUMERIC(14, 4) NOT NULL,
    precio_total              NUMERIC(16, 2) NOT NULL,
    diferencia_pedido_entrega INTEGER,
    estado_pedido             VARCHAR(20)
);

CREATE INDEX idx_compras_producto_id ON compras(producto_id);
CREATE INDEX idx_compras_fecha       ON compras(fecha);
CREATE INDEX idx_compras_factura     ON compras(factura);
"""

sql_vistas_bi = """
-- Calendario continuo generado en PostgreSQL para habilitar funciones DAX de tiempo en Power BI
CREATE OR REPLACE VIEW vw_bi_calendario AS
SELECT
    d::date AS fecha,
    EXTRACT(YEAR FROM d)::int AS anio,
    EXTRACT(QUARTER FROM d)::int AS trimestre,
    EXTRACT(MONTH FROM d)::int AS mes,
    TO_CHAR(d, 'YYYY-MM') AS anio_mes,
    EXTRACT(ISODOW FROM d)::int AS dia_semana,
    CASE EXTRACT(ISODOW FROM d)::int
        WHEN 1 THEN 'LUNES'
        WHEN 2 THEN 'MARTES'
        WHEN 3 THEN 'MIERCOLES'
        WHEN 4 THEN 'JUEVES'
        WHEN 5 THEN 'VIERNES'
        WHEN 6 THEN 'SABADO'
        WHEN 7 THEN 'DOMINGO'
    END AS nombre_dia,
    (EXTRACT(ISODOW FROM d) >= 6) AS es_fin_de_semana
FROM generate_series('2021-01-01'::date, '2026-12-31'::date, '1 day'::interval) AS d;

-- Catálogo de productos con indicador de movimiento histórico
CREATE OR REPLACE VIEW vw_bi_productos AS
SELECT
    p.id AS producto_id,
    p.codigo,
    p.nombre,
    p.droga,
    p.rubro,
    p.funcion,
    p.presentacion,
    p.laboratorio,
    p.activo AS es_catalogo_vigente,
    (pc.id IS NOT NULL) AS tiene_compras
FROM productos p
LEFT JOIN productos_comprados pc ON p.id = pc.id;

-- Tabla de hechos para BI; mantiene precios simbólicos en el cierre contable pero los marca para excluirlos en análisis de inflación
CREATE OR REPLACE VIEW vw_bi_compras AS
SELECT
    c.id AS compra_id,
    c.factura,
    LEFT(c.factura, 1) AS circuito,
    c.producto_id,
    c.fecha::date AS fecha,
    c.fecha AS fecha_hora,
    c.cant_ped,
    c.cant_fact,
    c.cant_entr,
    c.precio_unitario,
    c.precio_total,
    c.diferencia_pedido_entrega,
    c.estado_pedido,
    (c.precio_unitario > 1.00 AND c.cant_fact > 0) AS es_precio_mercado
FROM compras c;
"""

cols_catalogo = ["droga", "tipo", "accion", "clasificacion", "laboratorio"]

df_productos = pd.DataFrame({
    "codigo": pd.to_numeric(df_prod_raw["codigo"].str.strip(), errors="raise").astype("int64"),
    "nombre": df_prod_raw["producto"],
    "droga": df_prod_raw["droga"],
    "rubro": df_prod_raw["tipo"],
    "funcion": df_prod_raw["accion"],
    "presentacion": df_prod_raw["clasificacion"],
    "laboratorio": df_prod_raw["laboratorio"],
    "activo": ~df_prod_raw[cols_catalogo].isna().all(axis=1),
})

with engine.begin() as conn:
    conn.execute(text(sql_ddl_tablas))

    df_productos.to_sql("productos", conn, if_exists="append", index=False, chunksize=5000)

    mapa_ids = pd.read_sql(text("SELECT id, codigo FROM productos"), conn)
    mapa_codigo_a_id = dict(zip(mapa_ids["codigo"], mapa_ids["id"]))

    codigos_comp = pd.to_numeric(df_comp_raw["codigo"].str.strip(), errors="raise").astype("int64")
    diferencia = df_comp_raw["cant_ped"] - df_comp_raw["cant_entr"]

    df_compras = pd.DataFrame({
        "factura": df_comp_raw["factura"],
        "producto_id": codigos_comp.map(mapa_codigo_a_id),
        "codigo": codigos_comp,
        "fecha": pd.to_datetime(df_comp_raw["fecha_hora"]),
        "cant_ped": df_comp_raw["cant_ped"],
        "cant_fact": df_comp_raw["cant_fact"],
        "cant_entr": df_comp_raw["cant_entr"],
        "precio_unitario": df_comp_raw["precio_unitario"],
        "precio_total": df_comp_raw["importe_facturado"],
        "diferencia_pedido_entrega": diferencia,
        "estado_pedido": np.where(diferencia == 0, "COMPLETO", "PARCIAL"),
    })

    df_compras.to_sql("compras", conn, if_exists="append", index=False, chunksize=5000)

    df_prod_comprados = pd.DataFrame({"id": sorted(df_compras["producto_id"].unique())})
    df_prod_comprados.to_sql("productos_comprados", conn, if_exists="append", index=False)

    conn.execute(text(sql_vistas_bi))

In [11]:
with engine.connect() as conn:
    n_prod = conn.execute(text("SELECT COUNT(*) FROM productos")).scalar()
    n_activos = conn.execute(text("SELECT COUNT(*) FROM productos WHERE activo = TRUE")).scalar()
    n_comp_uniq = conn.execute(text("SELECT COUNT(*) FROM productos_comprados")).scalar()
    n_compras, total = conn.execute(text("SELECT COUNT(*), SUM(precio_total) FROM compras")).fetchone()
    n_mercado = conn.execute(text("SELECT COUNT(*) FROM vw_bi_compras WHERE es_precio_mercado")).scalar()

assert n_prod == 31_084, f"Error en productos: {n_prod}"
assert n_compras == 77_774, f"Error en compras: {n_compras}"
assert round(float(total), 2) == 566_760_205.92, f"Descuadre financiero: {total}"

print(f"productos              : {n_prod:,} ({n_activos:,} vigentes + {n_prod - n_activos:,} históricos)")
print(f"productos_comprados    : {n_comp_uniq:,} SKUs con compras")
print(f"compras                : {n_compras:,} renglones")
print(f"Total precio_total     : ${total:,.2f}")
print(f"Válidos precio mercado : {n_mercado:,} ({n_compras - n_mercado} líneas bonificadas)")

productos              : 31,084 (30,326 vigentes + 758 históricos)
productos_comprados    : 5,204 SKUs con compras
compras                : 77,774 renglones
Total precio_total     : $566,760,205.92
Válidos precio mercado : 77,724 (50 líneas bonificadas)
